In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parents[0]
PROCESSED_DIR = ROOT / "data" / "processed"

df = pd.read_parquet(PROCESSED_DIR / "sales_long.parquet")
df["date"] = pd.to_datetime(df["date"])

print("Loaded:", df.shape)


Loaded: (58327370, 22)


In [2]:
df = df.sort_values(["id", "date"])


In [3]:
lags = [1, 7, 28]

for lag in lags:
    df[f"lag_{lag}"] = df.groupby("id")["y"].shift(lag)


In [4]:
windows = [7, 28]

for w in windows:
    df[f"roll_mean_{w}"] = (
        df.groupby("id")["y"]
          .shift(1)
          .rolling(w)
          .mean()
          .reset_index(level=0, drop=True)
    )
    
    df[f"roll_std_{w}"] = (
        df.groupby("id")["y"]
          .shift(1)
          .rolling(w)
          .std()
          .reset_index(level=0, drop=True)
    )


In [5]:
df = df[df["lag_28"].notna()].copy()


In [6]:
print("Final shape:", df.shape)
print("Total columns:", len(df.columns))
print("Memory (GB):", df.memory_usage(deep=True).sum() / 1e9)


Final shape: (57473650, 29)
Total columns: 29
Memory (GB): 44.51988524


In [7]:
import numpy as np

def reduce_memory(df):
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object and str(col_type)[:8] != "category":
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == "int":
                if c_min >= 0:
                    if c_max < 255:
                        df[col] = df[col].astype(np.uint8)
                    elif c_max < 65535:
                        df[col] = df[col].astype(np.uint16)
                    else:
                        df[col] = df[col].astype(np.uint32)
                else:
                    if np.iinfo(np.int8).min < c_min < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif np.iinfo(np.int16).min < c_min < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    else:
                        df[col] = df[col].astype(np.int32)
            
            elif str(col_type)[:5] == "float":
                df[col] = df[col].astype(np.float32)
        
        else:
            df[col] = df[col].astype("category")
    
    return df

df = reduce_memory(df)

print("Memory after optimization (GB):", df.memory_usage(deep=True).sum() / 1e9)


Memory after optimization (GB): 4.257191033


In [8]:
df.to_parquet(PROCESSED_DIR / "features.parquet", index=False)
print("✅ saved:", PROCESSED_DIR / "features.parquet")


✅ saved: /Users/sohithmalyala/Desktop/Projects/demand-forecasting-platform/data/processed/features.parquet
